In [ ]:
import os
os.environ['OPENAI_API_KEY'] = "sk-proj-OR"

Install Libraries

In [ ]:
! pip install -q youtube-transcript-api langchain-community langchain-openai \ faiss-cpu tiktoken python-dotenv

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import PromtTemplate

Step 1a - Indexing (Document Ingestion)

In [ ]:
video_id = 'LPZh9BOjkQs' # only the ID, not full URL
try:
  # If you don't care which language , this returns the "best" one
    transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=['en'])

    # Flatten it to plain text
    transcript = " ".join(chunk['text'] for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No caption available for this video.")

In [ ]:
transcript_list

Step 1b - Indexing (Text Splitting)

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = splitter.create_documents([transcript])

In [ ]:
len (chunks)

In [ ]:
chunks[0]

Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [ ]:
embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")
vectorstore = FAISS.from_documents(chunks, embeddings)

In [ ]:
vectorstore.index_to_docstore_id

In [ ]:
vector_store.get_by_ids(['bc54_cec1'])

Step2 - Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 4})

In [ ]:
retriever

In [ ]:
retriever.invoke("What is deepmind")

Step3 - Argumentation

In [ ]:
llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0.2)

In [ ]:
prompt = PromptTemplate(
    template = """You are a helpful assistant.
    Answer ONLY from the provided transcript context.
    If the context is insufficient, just say you don't know.

    {context}

    Question: {question}
    """,
    input_variables=['context', 'question']
    )


In [ ]:
question = "is the topic af aliens discussed in the vedio? If yes then what was discussed"
retrieved_docs = retriever.invoke(question)


In [ ]:
retrieved_docs

In [ ]:
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

In [ ]:
context_text

In [ ]:
final_prompt = prompt.invoke("context":context_text, "question": question)

Step4-Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)

Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def forest_docs(retrieved_docs):
    context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])
    return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context' = retriever | RunnableLambda(forest_docs),
    'question' = RunnablePassthrough()
})


In [ ]:
parallel_chain.invoke("who is Demis")

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke ("Can you  summarize the video")